# raw_layer


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, BooleanType
transactions_schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("transaction_timestamp", TimestampType()),
    StructField("sender_vpa", StringType()),
    StructField("receiver_vpa", StringType()),
    StructField("receiver_merchant_id", StringType()),
    StructField("upi_app", StringType()),
    StructField("transaction_type", StringType()),
    StructField("amount", DoubleType()),
    StructField("status", StringType()),
    StructField("failure_reason", StringType()),
    StructField("bank_reference_number", StringType()),
    StructField("sender_bank", StringType()),
    StructField("receiver_bank", StringType()),
    StructField("state", StringType()),
    StructField("is_repeat_txn", BooleanType())
])

In [0]:
# dbutils.fs.rm("/Volumes/workspace/raw_upi/volumes/checkpoints/transactions", True)

True

In [0]:
#dbutils.fs.mkdirs("/Volumes/workspace/raw_upi/volumes/checkpoints/transactions")

True

In [0]:
from pyspark.sql.functions import *

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .schema(transactions_schema)
    .load("/Volumes/workspace/raw_upi/volumes/transactions/")
)

df = df \
    .withColumn("_ingest_timestamp", current_timestamp()) \
    .withColumn("_source_system", lit("upi_system")) \
    .withColumn("_batch_id", lit("batch_001")) \
    .withColumn("_file_name", col("_metadata.file_path"))

df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/workspace/raw_upi/volumes/checkpoints/transactions") \
    .trigger(availableNow=True) \
    .toTable("raw_upi.transactions_bronze")

In [0]:
%sql
-- SELECT status, COUNT(*) FROM workspace.raw_upi.transactions_bronze GROUP BY status;

status,COUNT(*)
SUCCESS,1756
PENDING,402
FAILED,842


In [0]:
# Create folder
#dbutils.fs.mkdirs("/Volumes/workspace/raw_upi/volumes/merchants")

# Move CSV file into that folder
#dbutils.fs.mv(
  #  "/Volumes/workspace/raw_upi/volumes/merchants.csv",
   # "/Volumes/workspace/raw_upi/volumes/merchants/merchants.csv"
#)

True

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, input_file_name, lit

merchants_schema = StructType([
    StructField("merchant_id", StringType()),
    StructField("business_name", StringType()),
    StructField("mcc_code", StringType()),
    StructField("merchant_category", StringType()),
    StructField("city", StringType()),
    StructField("state", StringType()),
    StructField("onboarding_date", DateType()),
    StructField("settlement_bank", StringType()),
    StructField("settlement_account", StringType()),
    StructField("is_active", BooleanType()),
    StructField("monthly_limit", DoubleType())
])

In [0]:
df_merchants = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .schema(merchants_schema) \
    .load("/Volumes/workspace/raw_upi/volumes/merchants")

In [0]:
df_merchants = df_merchants \
    .withColumn("_ingest_timestamp", current_timestamp()) \
    .withColumn("_source_system", lit("upi")) \
    .withColumn("_batch_id", lit("batch_001")) \
    .withColumn("_file_name", col("_metadata.file_path"))

In [0]:
df_merchants.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/workspace/raw_upi/volumes/checkpoints/merchants") \
    .trigger(availableNow=True) \
    .toTable("raw_upi.merchants_bronze")


In [0]:
# create folder
#dbutils.fs.mkdirs("/Volumes/workspace/raw_upi/volumes/bank_settlements")
# Move CSV file into that folder
#dbutils.fs.mv(
 #   "/Volumes/workspace/raw_upi/volumes/bank_settlements.csv",
  #  "/Volumes/workspace/raw_upi/volumes/bank_settlements/bank_settlements.csv",
   # True
#)

True

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, input_file_name, lit

bank_settlements_schema = StructType([
    StructField("settlement_id", StringType()),
    StructField("bank_reference_number", StringType()),
    StructField("settlement_date", DateType()),
    StructField("value_date", DateType()),
    StructField("bank_name", StringType()),
    StructField("merchant_id", StringType()),
    StructField("gross_amount", DoubleType()),
    StructField("charges_deducted", DoubleType()),
    StructField("net_settlement_amount", DoubleType()),
    StructField("settlement_status", StringType()),
    StructField("batch_id", StringType()),   
    StructField("remarks", StringType())
])

In [0]:
df_settlements = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .schema(bank_settlements_schema) \
    .load("/Volumes/workspace/raw_upi/volumes/bank_settlements")

In [0]:
df_settlements = df_settlements \
    .withColumn("_ingest_timestamp", current_timestamp()) \
    .withColumn("_source_system", lit("upi")) \
    .withColumn("_batch_id", lit("batch_001")) \
    .withColumn("_file_name", col("_metadata.file_path"))

In [0]:
df_settlements.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/workspace/raw_upi/volumes/checkpoints/bank_settlements") \
    .trigger(availableNow=True) \
    .toTable("workspace.raw_upi.bank_settlements_bronze")

In [0]:
# Create folder
#dbutils.fs.mkdirs("/Volumes/workspace/raw_upi/volumes/fraud_flags")

# Move CSV file into that folder
#dbutils.fs.mv(
 #   "/Volumes/workspace/raw_upi/volumes/fraud_flags.csv",
  #  "/Volumes/workspace/raw_upi/volumes/fraud_flags/fraud_flags.csv",
   # True)

True

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, input_file_name, lit

fraud_flags_schema = StructType([
    StructField("flag_id", StringType()),
    StructField("transaction_id", StringType()),
    StructField("flagged_at", TimestampType()),
    StructField("merchant_id", StringType()),
    StructField("sender_vpa", StringType()),
    StructField("fraud_rule", StringType()),
    StructField("risk_score", DoubleType()),
    StructField("disposition", StringType()),
    StructField("analyst_reviewed", BooleanType()),
    StructField("confirmed_fraud", BooleanType()),
    StructField("loss_amount", DoubleType())
])

In [0]:
df_fraud = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .schema(fraud_flags_schema) \
    .load("/Volumes/workspace/raw_upi/volumes/fraud_flags")

In [0]:
df_fraud = df_fraud \
    .withColumn("_ingest_timestamp", current_timestamp()) \
    .withColumn("_source_system", lit("upi")) \
    .withColumn("_batch_id", lit("batch_001")) \
    .withColumn("_file_name", col("_metadata.file_path"))

In [0]:
df_fraud.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/workspace/raw_upi/volumes/checkpoints/fraud_flags") \
    .trigger(availableNow=True) \
    .toTable("workspace.raw_upi.fraud_flags_bronze")

In [0]:
%sql
-- SELECT *  FROM workspace.raw_upi.fraud_flags_bronze LIMIT 100;

flag_id,transaction_id,flagged_at,merchant_id,sender_vpa,fraud_rule,risk_score,disposition,analyst_reviewed,confirmed_fraud,loss_amount,_ingest_timestamp,_source_system,_batch_id,_file_name
FRD000001,UPI00001513,2024-01-13T02:15:00.000Z,MER00488,user12837@bob,New device + new location,0.93,ALLOWED,null,null,0.0,2026-05-06T06:02:00.996Z,upi,batch_001,/Volumes/workspace/raw_upi/volumes/fraud_flags/fraud_flags.csv
FRD000002,UPI00001285,2024-02-16T06:44:00.000Z,MER00577,user21700@pnb,Mule account pattern,0.89,REVIEWED,null,null,0.0,2026-05-06T06:02:00.996Z,upi,batch_001,/Volumes/workspace/raw_upi/volumes/fraud_flags/fraud_flags.csv
FRD000003,UPI00001286,2024-02-13T05:33:00.000Z,MER00472,user74781@bob,Round-amount structuring,0.6,REVIEWED,null,null,0.0,2026-05-06T06:02:00.996Z,upi,batch_001,/Volumes/workspace/raw_upi/volumes/fraud_flags/fraud_flags.csv
FRD000004,UPI00001714,2024-03-25T14:30:00.000Z,MER00175,user14869@axis,Mule account pattern,0.67,ALLOWED,null,null,0.0,2026-05-06T06:02:00.996Z,upi,batch_001,/Volumes/workspace/raw_upi/volumes/fraud_flags/fraud_flags.csv
FRD000005,UPI00002458,2024-01-10T21:59:00.000Z,MER00372,user32332@axis,Cross-border attempt,0.62,BLOCKED,null,null,null,2026-05-06T06:02:00.996Z,upi,batch_001,/Volumes/workspace/raw_upi/volumes/fraud_flags/fraud_flags.csv
FRD000006,UPI00002310,2024-03-26T10:19:00.000Z,MER00085,user32433@hdfc,Blacklisted VPA,0.84,BLOCKED,null,null,null,2026-05-06T06:02:00.996Z,upi,batch_001,/Volumes/workspace/raw_upi/volumes/fraud_flags/fraud_flags.csv
FRD000007,UPI00001308,2024-02-25T07:04:00.000Z,MER00224,user87011@sbi,Round-amount structuring,0.71,BLOCKED,null,null,null,2026-05-06T06:02:00.996Z,upi,batch_001,/Volumes/workspace/raw_upi/volumes/fraud_flags/fraud_flags.csv
FRD000008,UPI00000896,2024-02-25T13:22:00.000Z,MER00328,user67293@icici,Cross-border attempt,0.66,REVIEWED,null,null,0.0,2026-05-06T06:02:00.996Z,upi,batch_001,/Volumes/workspace/raw_upi/volumes/fraud_flags/fraud_flags.csv
FRD000009,UPI00002243,2024-03-19T04:30:00.000Z,MER00562,user60036@icici,Round-amount structuring,0.65,BLOCKED,null,null,null,2026-05-06T06:02:00.996Z,upi,batch_001,/Volumes/workspace/raw_upi/volumes/fraud_flags/fraud_flags.csv
FRD000010,UPI00002072,2024-02-11T01:19:00.000Z,MER00249,user49925@kotak,High velocity � >5 txn in 10 min,0.74,ALLOWED,null,null,0.0,2026-05-06T06:02:00.996Z,upi,batch_001,/Volumes/workspace/raw_upi/volumes/fraud_flags/fraud_flags.csv
